# ClimateTwin: Exploratory Data Analysis & Modeling

This notebook performs Exploratory Data Analysis (EDA) on the processed Chennai weather, rainfall, reservoir levels, and flood events datasets, and trains/evaluates predictive models for flood likelihood.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_recall_curve, auc

processed_dir = '../data/processed'
print("Libraries loaded successfully.")

## 1. Load Data

In [ ]:
df_events = pd.read_csv(os.path.join(processed_dir, "flood_events", "chennai_flood_events.csv"))
df_imerg = pd.read_csv(os.path.join(processed_dir, "rainfall", "imerg_chennai_daily.csv"))
df_reservoirs = pd.read_csv(os.path.join(processed_dir, "water_level", "cmwssb_reservoir_levels.csv"))
df_noaa = pd.read_csv(os.path.join(processed_dir, "weather", "noaa_isd_chennai_hourly.csv"))

print(f"Flood events: {df_events.shape}")
print(f"IMERG Rainfall records: {df_imerg.shape}")
print(f"Reservoir Level records: {df_reservoirs.shape}")
print(f"NOAA Weather records: {df_noaa.shape}")

## 2. Preprocessing & Alignment

In [ ]:
# Convert datetime cols
df_imerg['time'] = pd.to_datetime(df_imerg['time'])
df_imerg.set_index('time', inplace=True)

df_reservoirs['date_standard'] = pd.to_datetime(df_reservoirs['date_standard'])
df_reservoirs = df_reservoirs.groupby('date_standard')[['POONDI', 'CHOLAVARAM', 'REDHILLS', 'CHEMBARAMBAKKAM']].mean()

df_noaa['DATE'] = pd.to_datetime(df_noaa['DATE'])
df_noaa.set_index('DATE', inplace=True)
df_noaa_daily = df_noaa.resample('1D').agg({
    'temperature_c': ['mean', 'max', 'min'],
    'dew_point_c': 'mean',
    'wind_speed_ms': 'mean'
})
df_noaa_daily.columns = ['temp_mean', 'temp_max', 'temp_min', 'dew_point_mean', 'wind_speed_mean']

print("Daily resampling completed.")

## 3. Engineering Target Variable (Active Flood Days)

In [ ]:
# Create target variable
intervals = []
for _, row in df_events.iterrows():
    try:
        start = pd.to_datetime(row['Start Date'])
        end = pd.to_datetime(row['End Date'])
        if pd.notna(start) and pd.notna(end):
            intervals.append((start, end))
    except:
        continue

start_date = max(df_imerg.index.min(), df_noaa_daily.index.min(), df_reservoirs.index.min())
end_date = min(df_imerg.index.max(), df_noaa_daily.index.max(), df_reservoirs.index.max())
print(f"Overlap time window: {start_date.date()} to {end_date.date()}")

date_range = pd.date_range(start_date, end_date, freq='D')
features = pd.DataFrame(index=date_range)

is_flood = np.zeros(len(date_range))
for i, dt in enumerate(date_range):
    for start, end in intervals:
        if start <= dt <= end:
            is_flood[i] = 1.0
            break
features['flood_active'] = is_flood

print(f"Target distribution: {features['flood_active'].value_counts()}")

## 4. Feature Engineering

In [ ]:
# Ingest IMERG, NOAA, and reservoirs
features['rainfall_daily'] = df_imerg['precipitation_mm_hr']
features['rainfall_3d_sum'] = features['rainfall_daily'].rolling(3, min_periods=1).sum()
features['rainfall_5d_sum'] = features['rainfall_daily'].rolling(5, min_periods=1).sum()
features['rainfall_7d_sum'] = features['rainfall_daily'].rolling(7, min_periods=1).sum()

features = features.join(df_noaa_daily, how='left')
features = features.join(df_reservoirs, how='left')

features.ffill(inplace=True)
features.bfill(inplace=True)

features['total_reservoir_level'] = features[['POONDI', 'CHOLAVARAM', 'REDHILLS', 'CHEMBARAMBAKKAM']].sum(axis=1)
print("Features engineered successfully.")

## 5. Exploratory Data Analysis & Correlations

In [ ]:
# Pearson correlations with target
corrs = features.corr()['flood_active'].sort_values(ascending=False)
print("Correlation with flood activity:")
print(corrs)

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(features.corr(), annot=True, fmt=".2f", cmap='coolwarm', cbar=True)
plt.title('Correlation Heatmap')
plt.show()

## 6. Time-based Split & Model Validation

In [ ]:
# Time-series split: train <= 2016-12-31, test >= 2017-01-01
split_date = pd.to_datetime('2016-12-31')
df_train = features[features.index <= split_date]
df_test = features[features.index > split_date]

X_train = df_train.drop(columns=['flood_active'])
y_train = df_train['flood_active']
X_test = df_test.drop(columns=['flood_active'])
y_test = df_test['flood_active']

print(f"Train size: {X_train.shape[0]}, positives: {int(y_train.sum())}")
print(f"Test size: {X_test.shape[0]}, positives: {int(y_test.sum())}")

# Train Random Forest & Gradient Boosting
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)

# Test metrics
rf_probs = rf.predict_proba(X_test)[:, 1]
gb_probs = gb.predict_proba(X_test)[:, 1]

print("Random Forest Test ROC-AUC:", roc_auc_score(y_test, rf_probs))
print("Gradient Boosting Test ROC-AUC:", roc_auc_score(y_test, gb_probs))

## 7. Walk-Forward Cross Validation

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
X_full = features.drop(columns=['flood_active'])
y_full = features['flood_active']

rf_cv_aucs = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_full)):
    y_te = y_full.iloc[test_idx]
    if y_te.sum() == 0: continue
    
    rf.fit(X_full.iloc[train_idx], y_full.iloc[train_idx])
    probs = rf.predict_proba(X_full.iloc[test_idx])[:, 1]
    rf_cv_aucs.append(roc_auc_score(y_te, probs))

print("Mean TimeSeriesSplit ROC-AUC (Random Forest):", np.mean(rf_cv_aucs))